# 05b — Treino do Random Forest binário (ajuste, validação, refit e custo computacional), variante COM os atributos de AUC alto

> **Variante COM os atributos de AUC alto.** Mesmos ajuste, validação e refit do notebook 05 (o código confere isso) e mesma configuração de hiperparâmetros: a **única** diferença são as colunas de entrada (`bwd_packets_s`, `inbound` e `total_backward_packets` voltam). Os artefatos vão para `artefatos_modelos/rf_binario_com_auc`. Todos os objetos criados aqui têm o sufixo `_com_auc`: **nada** da versão sem esses atributos é alterado.

Este notebook treina o **RF binário** (BENIGN × ataque) com a configuração inicial combinada, **mede o desempenho na validação** e **mede o custo computacional**. A busca de hiperparâmetros, o XAI e o SVM ficam para depois, e o **teste** é avaliado no notebook **06**, uma única vez.

## Protocolo

| Fase | Dados | O que acontece |
|---|---|---|
| **1. Ajuste** | `prep_vw_bal_bin_treino_com_auc` (81.228 linhas: 40.614 BENIGN + 40.614 ataques) | O RF é treinado |
| **Validação** | `prep_vw_bal_bin_validacao_com_auc` (60.154 linhas: 10.154 BENIGN + 50.000 ataques) | Métricas do modelo da fase 1, **dispersão em partições** e validação cruzada no ajuste |
| **2. Refit** | `prep_vw_bal_bin_refit_com_auc` (criada aqui: **50.768 BENIGN + 50.768 ataques = 101.536 linhas**) | O modelo **final** é refeito com todos os BENIGN. É ele que o notebook 06b avalia |
| **Teste** | notebook 06b | Usado uma vez |

**Como o refit é montado:** todo o ajuste + **todos** os BENIGN da validação + ataques da validação para manter o 1:1 (a mesma quantidade por tipo de ataque, sorteados com semente fixa). A seleção fica gravada em `prep_bal_bin_refit_sel_com_auc`, para o **SVM usar exatamente o mesmo conjunto** depois. Depois do refit, a validação **não mede mais nada** (os BENIGN dela entraram no treino): as métricas de qualidade vêm do modelo da fase 1, e o modelo final é avaliado só no teste.

## Como ler as métricas

| Métrica | O que mede | Depende da proporção BENIGN:ataque? |
|---|---|---|
| **ROC-AUC** | Quão bem o modelo ordena ataques acima de BENIGN (0,5 = acaso; 1 = perfeito) | Não |
| **Recall** (ataques) | Fração dos ataques detectados | Não |
| **Taxa de falso alarme** | Fração dos BENIGN chamados de ataque | Não |
| **Especificidade**, **acurácia balanceada** | 1 − falso alarme; média de recall e especificidade | Não |
| **Acurácia** | Fração de acertos | **Sim** |
| **Precisão** | Dos alarmes, quantos eram ataques de fato | **Sim** |
| **F1** | Média harmônica de precisão e recall | **Sim** |
| **MCC** | Correlação entre previsão e verdade (−1 a 1); robusta a classes desbalanceadas | **Sim** |
| **PR-AUC** | Área sob a curva precisão × recall | **Sim** |
| **Precisão, F1 e PR-AUC do BENIGN** (`precisao_benign`, `f1_benign`, `pr_auc_benign`) e **F1 macro** | As mesmas contas com o **BENIGN** como classe "positiva" (e a média dos dois F1) | **Sim** |

**Por que olhar também o BENIGN?** Com ~99% de ataques, a precisão, o F1 e o PR-AUC **dos ataques** ficam perto de 1 mesmo para um modelo fraco, porque quase tudo é ataque. As métricas da classe rara (BENIGN) e o MCC são as que de fato informam. A linha de base do PR-AUC do BENIGN é a sua proporção (~0,6%).

**Duas colunas de resultado.** A validação tem 17% de BENIGN, e o mundo real tem ~0,6%. Por isso as métricas que **dependem da proporção** são mostradas também com um peso que a corrige **sem favorecer nenhum tipo de ataque**: ataques pesam 1 e cada BENIGN pesa ≈ 0,0305, o que devolve a proporção real do treino (0,615%). **Para as métricas que não dependem da proporção as duas colunas são iguais.** A coluna `peso` da view de validação **não é usada** (ela faria o TFTP valer 54%).

**Operating points.** Além do limiar 0,5, mostramos o **recall com falso alarme fixo** (1% e 0,1%): o limiar é escolhido **na validação** e fica gravado para o teste.

## Custo computacional (medido durante o treino)

| Medida | Como |
|---|---|
| **Tempo de treino** | Relógio (`perf_counter`) e **tempo de CPU** (o RF usa vários núcleos, então o relógio sozinho esconde isso). O refit é repetido `REPETICOES_TEMPO` vezes (mediana, mínimo e máximo) |
| **Memória** | **Pico de RAM** do processo durante o treino (amostrado a cada 50 ms com `psutil`), tamanho do modelo em disco e número de nós das árvores |
| **Inferência** | **Latência** de 1 fluxo por vez (p50, p95, p99) e **vazão** em lote (fluxos/s), com `n_jobs=1` e com todos os núcleos, **com e sem o pré-processamento** (troca dos vazios pela mediana e indicadores). A leitura do banco fica de fora |

## O que é salvo (pasta `artefatos_modelos/rf_binario_com_auc/rf_bin_<data>`)

`modelo_ajuste.joblib`, `modelo_final_refit.joblib`, `config_treino.json` (hiperparâmetros, sementes, versões, dados, software e hardware), `preprocessamento.json` (**nomes e ordem dos atributos**, medianas, indicadores, colunas retiradas), `metricas_validacao.json/csv`, `dispersao_particoes_validacao.csv`, `cv_ajuste.csv`, `custo_computacional.json/csv`, `limiares_validacao.json`, `refit_ids.csv`, e uma linha nova em `registro_experimentos.csv` (um registro por execução, base para a futura busca de hiperparâmetros).

## Antes de rodar
Os notebooks 02, 03b e 04b precisam ter terminado. Coloque este arquivo na **mesma pasta** deles (onde está o `.env`). Bibliotecas: `scikit-learn`, `joblib` e `psutil`.

In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
def _div(a, b): return float(a) / float(b) if b else float("nan")

def metricas(y, p, limiar=0.5, w=None):
    # Todas as métricas de uma vez. Com `w` (pesos), as contagens e as métricas que dependem da proporção ficam ponderadas.
    # O ROC-AUC é calculado sem pesos: ele só compara ataques com BENIGN e não depende da proporção.
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float); pred = (p >= limiar).astype(int)
    w_ = np.ones(len(y)) if w is None else np.asarray(w, dtype=float)
    tp = float(w_[(y == 1) & (pred == 1)].sum()); tn = float(w_[(y == 0) & (pred == 0)].sum())
    fp = float(w_[(y == 0) & (pred == 1)].sum()); fn = float(w_[(y == 1) & (pred == 0)].sum())
    rec, esp, prec = _div(tp, tp + fn), _div(tn, tn + fp), _div(tp, tp + fp)
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    f1 = _div(2 * prec * rec, prec + rec) if (prec == prec and rec == rec) else float("nan")
    prec_b = _div(tn, tn + fn); f1_b = _div(2 * prec_b * esp, prec_b + esp) if (prec_b == prec_b and esp == esp) else float("nan")    # classe BENIGN como "positiva"
    return {"acuracia": _div(tp + tn, tp + tn + fp + fn), "acuracia_balanceada": (rec + esp) / 2, "precisao": prec, "recall": rec, "especificidade": esp,
            "taxa_falso_alarme": 1 - esp, "taxa_perda": 1 - rec, "f1": f1, "mcc": _div(tp * tn - fp * fn, den),
            "precisao_benign": prec_b, "f1_benign": f1_b, "f1_macro": (f1 + f1_b) / 2, "pr_auc_benign": float(average_precision_score(1 - y, 1 - p, sample_weight=w_)),
            "roc_auc": float(roc_auc_score(y, p)), "pr_auc": float(average_precision_score(y, p, sample_weight=w_)), "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def recall_em_fpr(y, p, alvo):
    # Ponto de operação: o MAIOR recall com taxa de falso alarme <= alvo (limiar escolhido nestes dados).
    fpr, tpr, thr = roc_curve(np.asarray(y).astype(int), np.asarray(p, dtype=float))
    i = max(int(np.searchsorted(fpr, alvo, side="right")) - 1, 0)
    lim = float(thr[i]) if np.isfinite(thr[i]) else 1.0 + 1e-9
    return {"alvo_fpr": alvo, "limiar": lim, "recall": float(tpr[i]), "fpr": float(fpr[i])}

class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# autoteste das ferramentas (não toca no servidor)
_y = np.array([0, 0, 0, 0, 1, 1, 1, 1]); _p = np.array([.1, .2, .3, .6, .4, .7, .8, .9]); _m = metricas(_y, _p, 0.5)
assert (_m["tp"], _m["fp"], _m["tn"], _m["fn"]) == (3, 1, 3, 1) and abs(_m["mcc"] - 0.5) < 1e-12 and abs(_m["acuracia"] - 0.75) < 1e-12, _m
_o = recall_em_fpr(_y, _p, 0.25); assert abs(_o["recall"] - 1.0) < 1e-12 and _o["fpr"] <= 0.25 and abs(_o["limiar"] - 0.4) < 1e-12, _o
_m2 = metricas(_y, _p, 0.5); assert abs(_m2["precisao_benign"] - 0.75) < 1e-12 and abs(_m2["f1_benign"] - 0.75) < 1e-12 and abs(_m2["f1_macro"] - 0.75) < 1e-12, _m2
_o0 = recall_em_fpr(_y, _p, 0.0); assert abs(_o0["recall"] - 0.75) < 1e-12 and _o0["fpr"] == 0.0, _o0
_d = distribuir(100, {"a": 1000, "b": 1000, "c": 1000, "d": 10}, 0.0, {"a": 1000, "b": 1000, "c": 1000, "d": 10}); assert _d["d"] == 10 and sum(_d.values()) == 100, _d
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2


## 1. Parâmetros e o que existe

- `HIPERPARAMETROS`: a **configuração inicial** combinada. A busca de hiperparâmetros vem depois.
- `K_PARTICOES_VAL`: a validação é dividida em K partições disjuntas e estratificadas, e as métricas são calculadas em cada uma, para mostrar a **dispersão** (quanto o resultado varia de uma parte da validação para outra). Isso **não** retreina o modelo.
- `K_CV`: validação cruzada **no ajuste** (K treinos com 4/5 do ajuste), para mostrar também a variação causada pelos dados de treino. Use 0 para pular.
- `REPETICOES_TEMPO`: quantas vezes o refit é repetido para medir o tempo (o modelo guardado é o da 1ª repetição; as outras devem dar o **mesmo** modelo, o que é conferido).
- `PESO_BENIGN_PROP`: calculado aqui a partir das contagens (≈ 0,0305 com os seus dados), para a validação ponderada ter a proporção real de BENIGN.
- `RAZAO_ATAQUE_BENIGNO`: a mesma do notebook 04b (1,0 = balanceado).

In [6]:
# ---------- 1.1 Parâmetros ----------
HIPERPARAMETROS = globals().get("HIPERPARAMETROS", dict(n_estimators=100, max_depth=None, min_samples_split=2, min_samples_leaf=1, max_features="sqrt", random_state=42, n_jobs=-1))
SEMENTE = globals().get("SEMENTE", 42)                       # partições, validação cruzada, refit e amostras de latência
RAZAO_ATAQUE_BENIGNO = globals().get("RAZAO_ATAQUE_BENIGNO", 1.0)
K_PARTICOES_VAL = globals().get("K_PARTICOES_VAL", 5)
K_CV = globals().get("K_CV", 5)
REPETICOES_TEMPO = globals().get("REPETICOES_TEMPO", 3)
N_LATENCIA = globals().get("N_LATENCIA", 1000)
FPR_ALVOS = globals().get("FPR_ALVOS", (0.01, 0.001))
LIMIAR = globals().get("LIMIAR", 0.5)
COMPRESSAO_MODELO = globals().get("COMPRESSAO_MODELO", 3)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
ARTEFATOS_BASE = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos" / "rf_binario_com_auc"
PASTA = ARTEFATOS_BASE / f"rf_bin_{RUN_ID}"; PASTA.mkdir(parents=True, exist_ok=True)

V_AJUSTE, V_VALID, V_REFIT = "prep_vw_bal_bin_treino_com_auc", "prep_vw_bal_bin_validacao_com_auc", "prep_vw_bal_bin_refit_com_auc"
SEL_TAB, DADOS_TAB, TAB_REFIT = "prep_bal_bin_selecao_com_auc", "prep_bal_bin_dados_com_auc", "prep_bal_bin_refit_sel_com_auc"
faltam = [t for t in ("prep_final_treino", V_AJUSTE, V_VALID, SEL_TAB, DADOS_TAB, "prep_views_rf_binario_com_auc") if not existe(t)]
assert not faltam, f"Rode antes os notebooks 02, 03b e 04b: faltam {faltam}"

versao_sel = str(consultar(f"SELECT any(versao) AS v FROM {SEL_TAB}")["v"][0])
aud = consultar_num("SELECT posicao, feature, situacao, mediana_imputacao, tem_indicador, versao FROM prep_views_rf_binario_com_auc ORDER BY posicao")
versao_views = str(aud["versao"].iloc[0])
N_BENIGN, N_ATAQUE = n_linhas("prep_final_treino", "label_binary = 0"), n_linhas("prep_final_treino", "label_binary = 1")
cont = consultar_num(f"SELECT papel, y, count() AS n FROM {SEL_TAB} GROUP BY papel, y"); cont = {(r.papel, int(r.y)): int(r.n) for r in cont.itertuples()}
n_aj_b, n_aj_a, n_va_b, n_va_a = cont[("treino", 0)], cont[("treino", 1)], cont[("validacao", 0)], cont[("validacao", 1)]
PESO_BENIGN_PROP = (N_BENIGN / N_ATAQUE) * (n_va_a / n_va_b)
print(f"Pasta dos artefatos: {PASTA}")
print(f"Seleção do notebook 04b (versão {versao_sel}) | views do notebook 03b (versão {versao_views})")
print(f"Treino: {N_BENIGN:,} BENIGN e {N_ATAQUE:,} ataques | ajuste: {n_aj_b:,} + {n_aj_a:,} | validação: {n_va_b:,} + {n_va_a:,}")
print(f"Peso de cada BENIGN na validação 'proporção real': {PESO_BENIGN_PROP:.5f} (ataques = 1)")
print("Hiperparâmetros:", HIPERPARAMETROS)

Pasta dos artefatos: C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_binario_com_auc\rf_bin_20261024_002027
Seleção do notebook 04b (versão 20261024_001450) | views do notebook 03b (versão 20261024_001223)
Treino: 50,768 BENIGN e 8,206,575 ataques | ajuste: 40,614 + 40,614 | validação: 10,154 + 50,000
Peso de cada BENIGN na validação 'proporção real': 0.03046 (ataques = 1)
Hiperparâmetros: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'random_state': 42, 'n_jobs': -1}


## 2. Conjunto do refit

O refit contém **todos os BENIGN**, **todo o ajuste** e os ataques que faltam para manter o 1:1. Esses ataques vêm da **validação**, na **mesma quantidade por tipo** (os tipos pequenos entram até onde houver linhas e o que falta é repartido entre os demais), sorteados com semente fixa. O `tipo` é usado **só para sortear**; ele não é atributo do modelo.

A seleção é gravada em `prep_bal_bin_refit_sel_com_auc`, e a view `prep_vw_bal_bin_refit_com_auc` (`id_linha`, `y` e atributos) é criada a partir de `prep_bal_bin_dados_com_auc`. Se esta célula for repetida, o resultado é o mesmo.

In [7]:
# ---------- 2.1 Montagem e gravação do refit ----------
sel = consultar(f"SELECT toString(id_linha) AS id, papel, y, tipo FROM {SEL_TAB}"); sel["y"] = sel["y"].astype(int)
aj_sel, va_sel = sel[sel.papel == "treino"], sel[sel.papel == "validacao"]
extra_total = int(round(RAZAO_ATAQUE_BENIGNO * N_BENIGN)) - int((aj_sel.y == 1).sum())
va_att = va_sel[va_sel.y == 1]; por_tipo = va_att.groupby("tipo").size().to_dict()
assert 0 <= extra_total <= sum(por_tipo.values()), f"a validação tem {sum(por_tipo.values()):,} ataques e o refit precisa de {extra_total:,}: aumente VALIDACAO_ATAQUES no notebook 04b"
quotas = distribuir(extra_total, por_tipo, 0.0, por_tipo)
extras = []
for t, n_t in quotas.items():
    if n_t <= 0: continue
    ids_t = sorted(va_att.loc[va_att.tipo == t, "id"], key=int)
    extras += [ids_t[k] for k in np.random.default_rng([SEMENTE, zlib.crc32(t.encode())]).permutation(len(ids_t))[:n_t]]
refit_sel = pd.concat([aj_sel.assign(origem="ajuste"), va_sel[va_sel.y == 0].assign(origem="validacao_benign"), va_att[va_att.id.isin(set(extras))].assign(origem="validacao_ataque_extra")], ignore_index=True)
assert refit_sel.id.is_unique and refit_sel.id.map(str.isdigit).all()

executar(f"DROP TABLE IF EXISTS `{TAB_REFIT}`")
executar(f"CREATE TABLE {TAB_REFIT} (id_linha UInt64, origem String, tipo String, versao String) ENGINE = MergeTree ORDER BY id_linha")
for i in range(0, len(refit_sel), 20_000):
    p = refit_sel.iloc[i:i + 20_000]
    executar_robusto(f"INSERT INTO {TAB_REFIT} VALUES " + ", ".join(f"({r.id}, {lit(r.origem)}, {lit(r.tipo)}, {lit(RUN_ID)})" for r in p.itertuples()))
cols_modelo_sql = ", ".join(q(c) for c in colunas_view(V_AJUSTE) if c not in ("id_linha", "y"))
executar(f"DROP VIEW IF EXISTS `{V_REFIT}`")
executar(f"CREATE VIEW {V_REFIT} AS SELECT id_linha, y, {cols_modelo_sql} FROM {DADOS_TAB} WHERE id_linha IN (SELECT id_linha FROM {TAB_REFIT})")
refit_sel[["id", "origem", "tipo", "y"]].rename(columns={"id": "id_linha"}).to_csv(PASTA / "refit_ids.csv", index=False)
print(f"✔ {TAB_REFIT}: {n_linhas(TAB_REFIT):,} linhas | view {V_REFIT}")
print(f"  ataques acrescentados vindos da validação: {extra_total:,} ({len([t for t, n in quotas.items() if n > 0])} tipos; quantidade por tipo: min {min(quotas.values()):,}, máx {max(quotas.values()):,})")

✔ prep_bal_bin_refit_sel_com_auc: 101,536 linhas | view prep_vw_bal_bin_refit_com_auc
  ataques acrescentados vindos da validação: 10,154 (12 tipos; quantidade por tipo: min 83, máx 916)


## 3. Pré-processamento e leitura dos dados

O pré-processamento do RF é **o da view definitiva** (notebook 03b): vazios trocados pela mediana do **treino** e uma coluna `<coluna>_ausente` (0/1) nas colunas que têm vazios, sem as colunas da ablação. Aqui ele é **também guardado como código** (`PreprocessadorRF`), para ser aplicado a fluxos novos: os nomes e a **ordem** das colunas e as medianas vão para `preprocessamento.json`.

Conferência: reconstruímos "fluxos crus" (com `NaN` onde o indicador vale 1) a partir da validação e passamos pelo `PreprocessadorRF`; o resultado tem de ser **idêntico** à matriz das views.

In [8]:
# ---------- 3.1 Pré-processamento e dados ----------
USADAS, RETIRADAS = aud[aud.situacao == "usada"], aud[aud.situacao != "usada"].feature.tolist()
MEDIANAS = {r.feature: float(r.mediana_imputacao) for r in USADAS.itertuples() if int(r.tem_indicador) == 1}
prep = PreprocessadorRF(USADAS.feature.tolist(), MEDIANAS, RETIRADAS, versao_views)
assert prep.colunas_modelo == [c for c in colunas_view(V_AJUSTE) if c not in ("id_linha", "y")], "a ordem das colunas da view difere da do pré-processamento"
print(f"Atributos do modelo: {len(prep.colunas_modelo)} ({len(prep.features)} + {len(prep.indicadores)} indicadores) | retiradas na ablação: {RETIRADAS}")

aj, va, rf_df = ler_view(V_AJUSTE), ler_view(V_VALID), ler_view(V_REFIT)
CM = prep.colunas_modelo
X_aj, y_aj = aj[CM].to_numpy(float), aj["y"].to_numpy(int)
X_va, y_va = va[CM].to_numpy(float), va["y"].to_numpy(int)
X_rf, y_rf = rf_df[CM].to_numpy(float), rf_df["y"].to_numpy(int)
for nome, X in (("ajuste", X_aj), ("validação", X_va), ("refit", X_rf)): assert np.isfinite(X).all(), f"valores não finitos em {nome}"

X_va_raw = X_va[:, :len(prep.features)].copy()
for j, c in enumerate(prep.indicadores): X_va_raw[X_va[:, len(prep.features) + j] == 1, prep.features.index(c)] = np.nan
assert np.allclose(prep.transformar(X_va_raw), X_va, rtol=1e-12, atol=1e-12), "o PreprocessadorRF não reproduz a matriz da view"
print(f"✔ PreprocessadorRF reproduz a matriz das views ({len(X_va):,} linhas com {int(np.isnan(X_va_raw).any(axis=1).sum()):,} fluxos com algum valor ausente)")
print(f"Ajuste: {len(X_aj):,} linhas ({int((y_aj == 0).sum()):,} BENIGN) | validação: {len(X_va):,} ({int((y_va == 0).sum()):,} BENIGN) | refit: {len(X_rf):,} ({int((y_rf == 0).sum()):,} BENIGN)")

Atributos do modelo: 46 (45 + 1 indicadores) | retiradas na ablação: []
✔ PreprocessadorRF reproduz a matriz das views (60,154 linhas com 1,132 fluxos com algum valor ausente)
Ajuste: 81,228 linhas (40,614 BENIGN) | validação: 60,154 (10,154 BENIGN) | refit: 101,536 (50,768 BENIGN)


## 4. Fase 1 — treino no ajuste e avaliação na validação

O modelo é treinado **só no ajuste** (com o tempo, a CPU e o pico de memória medidos) e avaliado na validação. A tabela mostra cada métrica **sem pesos** e com a **proporção real**. Em seguida:
- **operating points:** recall com falso alarme de 1% e de 0,1% (limiar escolhido na validação);
- **dispersão:** as mesmas métricas em K partições da validação (média, desvio, mínimo e máximo);
- **validação cruzada no ajuste:** K treinos, cada um avaliado na parte que ficou de fora.

In [9]:
# ---------- 4.1 Treino no ajuste ----------
with Medidor() as med_aj:
    rf = RandomForestClassifier(**HIPERPARAMETROS).fit(X_aj, y_aj)
rf.colunas_modelo_ = list(CM)                                    # a ordem dos atributos viaja dentro do arquivo do modelo
fit_ajuste = {"relogio_s": med_aj.relogio, "cpu_s": med_aj.cpu, "pico_ram_mib": med_aj.pico_mib, "acrescimo_ram_mib": med_aj.delta_mib}
print(f"✔ RF treinado no ajuste: {med_aj.relogio:.1f} s de relógio, {med_aj.cpu:.1f} s de CPU, pico de RAM {med_aj.pico_mib:,.0f} MiB (+{med_aj.delta_mib:,.0f} MiB sobre o início)")

# ---------- 4.2 Métricas na validação ----------
p_va = rf.predict_proba(X_va)[:, 1]
w_prop = np.where(y_va == 0, PESO_BENIGN_PROP, 1.0)
m_sem, m_real = metricas(y_va, p_va, LIMIAR, None), metricas(y_va, p_va, LIMIAR, w_prop)
DEPENDE = {k: "sim" for k in ("acuracia", "precisao", "f1", "mcc", "pr_auc", "precisao_benign", "f1_benign", "f1_macro", "pr_auc_benign")}
ordem = ["roc_auc", "recall", "taxa_falso_alarme", "especificidade", "acuracia_balanceada", "acuracia", "precisao", "f1", "mcc", "pr_auc", "precisao_benign", "f1_benign", "f1_macro", "pr_auc_benign"]
tab_val = pd.DataFrame({"sem pesos": {k: m_sem[k] for k in ordem}, "proporção real": {k: m_real[k] for k in ordem}})
tab_val["depende da proporção?"] = [DEPENDE.get(k, "não") for k in tab_val.index]
print(f"\n== Métricas na validação (limiar {LIMIAR}) ==  (para as métricas que 'não' dependem da proporção, as duas colunas são iguais)")
display(tab_val.round(5))
print("Matriz de confusão (sem pesos): VP={tp:,.0f} FP={fp:,.0f} VN={tn:,.0f} FN={fn:,.0f}".format(**m_sem))

PONTOS = []
for alvo in FPR_ALVOS:
    op = recall_em_fpr(y_va, p_va, alvo); mo = metricas(y_va, p_va, op["limiar"], w_prop)
    PONTOS.append({"alvo_fpr": alvo, "limiar": op["limiar"], "recall": op["recall"], "fpr_obtida": op["fpr"], "precisao_real": mo["precisao"], "f1_real": mo["f1"], "mcc_real": mo["mcc"], "precisao_benign_real": mo["precisao_benign"], "f1_benign_real": mo["f1_benign"]})
tab_pontos = pd.DataFrame(PONTOS); print("\n== Ponto de operação: recall com falso alarme fixo (limiar escolhido na validação) =="); display(tab_pontos.round(5))

# ---------- 4.3 Dispersão entre partições da validação ----------
linhas_part = []
for k, (_, idx) in enumerate(StratifiedKFold(K_PARTICOES_VAL, shuffle=True, random_state=SEMENTE).split(X_va, y_va), 1):
    m1, m2 = metricas(y_va[idx], p_va[idx], LIMIAR, None), metricas(y_va[idx], p_va[idx], LIMIAR, w_prop[idx])
    linhas_part.append({"particao": k, "n": len(idx), "benign": int((y_va[idx] == 0).sum()), "roc_auc": m1["roc_auc"], "recall": m1["recall"], "taxa_falso_alarme": m1["taxa_falso_alarme"],
                        "recall_fpr1pct": recall_em_fpr(y_va[idx], p_va[idx], 0.01)["recall"], "precisao_real": m2["precisao"], "f1_real": m2["f1"], "mcc_real": m2["mcc"], "pr_auc_real": m2["pr_auc"]})
tab_part = pd.DataFrame(linhas_part)
resumo_part = tab_part.drop(columns=["particao", "n", "benign"]).agg(["mean", "std", "min", "max"]).T
print(f"\n== Dispersão entre {K_PARTICOES_VAL} partições da validação =="); display(tab_part.round(5)); display(resumo_part.round(5))

# ---------- 4.4 Validação cruzada no ajuste ----------
linhas_cv = []
if K_CV and K_CV > 1:
    for k, (itr, ite) in enumerate(StratifiedKFold(K_CV, shuffle=True, random_state=SEMENTE).split(X_aj, y_aj), 1):
        t0 = time.perf_counter(); mk = RandomForestClassifier(**HIPERPARAMETROS).fit(X_aj[itr], y_aj[itr]); tf = time.perf_counter() - t0
        mk_ = metricas(y_aj[ite], mk.predict_proba(X_aj[ite])[:, 1], LIMIAR, None)
        linhas_cv.append({"dobra": k, "n_treino": len(itr), "n_avaliacao": len(ite), "tempo_treino_s": tf, "roc_auc": mk_["roc_auc"], "recall": mk_["recall"], "taxa_falso_alarme": mk_["taxa_falso_alarme"],
                          "acuracia_balanceada": mk_["acuracia_balanceada"], "mcc": mk_["mcc"]})
    tab_cv = pd.DataFrame(linhas_cv); resumo_cv = tab_cv.drop(columns=["dobra", "n_treino", "n_avaliacao"]).agg(["mean", "std", "min", "max"]).T
    print(f"\n== Validação cruzada no ajuste ({K_CV} dobras; cada dobra é balanceada 1:1) =="); display(tab_cv.round(5)); display(resumo_cv.round(5))
else:
    tab_cv, resumo_cv = pd.DataFrame(), pd.DataFrame(); print("\n(validação cruzada no ajuste desligada: K_CV = 0)")

✔ RF treinado no ajuste: 3.1 s de relógio, 21.6 s de CPU, pico de RAM 591 MiB (+29 MiB sobre o início)

== Métricas na validação (limiar 0.5) ==  (para as métricas que 'não' dependem da proporção, as duas colunas são iguais)


,sem pesos,proporção real,depende da proporção?
roc_auc,0.99994,0.99994,não
recall,0.99944,0.99944,não
taxa_falso_alarme,0.00030,0.00030,não
especificidade,0.99970,0.99970,não
acuracia_balanceada,0.99957,0.99957,não
acuracia,0.99948,0.99944,sim
precisao,0.99994,1.00000,sim
f1,0.99969,0.99972,sim
mcc,0.99817,0.95717,sim
pr_auc,0.99998,1.00000,sim


Matriz de confusão (sem pesos): VP=49,972 FP=3 VN=10,151 FN=28

== Ponto de operação: recall com falso alarme fixo (limiar escolhido na validação) ==


,alvo_fpr,limiar,recall,fpr_obtida,precisao_real,f1_real,mcc_real,precisao_benign_real,f1_benign_real
0,0.010,0.04,0.99998,0.00886,0.99995,0.99996,0.99390,0.99675,0.99393
1,0.001,0.23,0.99982,0.00089,0.99999,0.99991,0.98522,0.97170,0.98522



== Dispersão entre 5 partições da validação ==


,particao,n,benign,roc_auc,recall,taxa_falso_alarme,recall_fpr1pct,precisao_real,f1_real,mcc_real,pr_auc_real
0,1,12031,2031,0.99995,0.9996,0.00000,0.9999,1.00000,0.9998,0.96897,1.0
1,2,12031,2031,1.00000,0.9994,0.00049,1.0000,1.00000,0.9997,0.95423,1.0
2,3,12031,2031,1.00000,0.9994,0.00000,1.0000,1.00000,0.9997,0.95449,1.0
3,4,12031,2031,1.00000,0.9996,0.00000,1.0000,1.00000,0.9998,0.96897,1.0
4,5,12030,2030,0.99975,0.9992,0.00099,1.0000,0.99999,0.9996,0.94009,1.0


,mean,std,min,max
roc_auc,0.99994,0.00011,0.99975,1.00000
recall,0.99944,0.00017,0.99920,0.99960
taxa_falso_alarme,0.00030,0.00044,0.00000,0.00099
recall_fpr1pct,0.99998,0.00004,0.99990,1.00000
precisao_real,1.00000,0.00000,0.99999,1.00000
f1_real,0.99972,0.00008,0.99960,0.99980
mcc_real,0.95735,0.01210,0.94009,0.96897
pr_auc_real,1.00000,0.00000,1.00000,1.00000



== Validação cruzada no ajuste (5 dobras; cada dobra é balanceada 1:1) ==


,dobra,n_treino,n_avaliacao,tempo_treino_s,roc_auc,recall,taxa_falso_alarme,acuracia_balanceada,mcc
0,1,64982,16246,4.11156,1.00000,0.99889,0.00000,0.99945,0.99889
1,2,64982,16246,4.63450,0.99993,0.99926,0.00025,0.99951,0.99902
2,3,64982,16246,4.74601,0.99987,0.99902,0.00062,0.99920,0.99840
3,4,64983,16245,4.79219,1.00000,0.99877,0.00062,0.99908,0.99815
4,5,64983,16245,4.93755,0.99999,0.99938,0.00049,0.99945,0.99889


,mean,std,min,max
tempo_treino_s,4.64436,0.31706,4.11156,4.93755
roc_auc,0.99996,0.00006,0.99987,1.00000
recall,0.99906,0.00026,0.99877,0.99938
taxa_falso_alarme,0.00039,0.00027,0.00000,0.00062
acuracia_balanceada,0.99934,0.00019,0.99908,0.99951
mcc,0.99867,0.00037,0.99815,0.99902


## 5. Custo computacional do modelo da fase 1

- **Tamanho e estrutura:** arquivo em disco, número de árvores e de nós, e profundidade.
- **Latência:** tempo para classificar **1 fluxo por vez** (p50, p95 e p99 de 1.000 fluxos), com `n_jobs=1` e com todos os núcleos. "Fim a fim" inclui o pré-processamento (trocar vazios pela mediana e criar os indicadores). A leitura do banco **não** entra.
- **Vazão:** fluxos por segundo ao classificar a validação inteira de uma vez.

Com todos os núcleos, a latência de **um** fluxo costuma ser **pior** que com 1 núcleo (o custo de dividir o trabalho é maior que o ganho), e a vazão em lote **melhora**. Os dois números importam, para casos de uso diferentes.

In [10]:
# ---------- 5.1 Custo do modelo da fase 1 ----------
tam_aj = tamanho_e_nos(rf, PASTA / "modelo_ajuste.joblib", COMPRESSAO_MODELO)
inf_aj = medir_inferencia(rf, prep, X_va, X_va_raw, n_lat=N_LATENCIA, semente=SEMENTE)
print(f"Modelo do ajuste: {tam_aj['arquivo_mib']:.1f} MiB em disco | {tam_aj['n_arvores']} árvores | {tam_aj['nos_total']:,} nós | profundidade média {tam_aj['profundidade_media']:.1f} (máx {tam_aj['profundidade_max']})")
display(pd.DataFrame(inf_aj).set_index("n_jobs").T.round(3))

Modelo do ajuste: 0.7 MiB em disco | 100 árvores | 35,274 nós | profundidade média 18.5 (máx 23)


n_jobs,1,-1
n_latencia,1000.000,1000.000
latencia_modelo_p50_ms,9.477,52.142
latencia_modelo_p95_ms,20.147,103.493
latencia_modelo_p99_ms,27.623,118.467
latencia_fim_a_fim_p50_ms,9.555,52.287
latencia_fim_a_fim_p95_ms,20.226,104.120
latencia_fim_a_fim_p99_ms,27.828,118.637
vazao_modelo_fluxos_s,140878.749,340547.092
vazao_fim_a_fim_fluxos_s,124153.865,223932.409
n_lote,60154.000,60154.000


## 6. Fase 2 — refit (modelo final)

O modelo é refeito com a **mesma configuração** e todo o conjunto do refit, `REPETICOES_TEMPO` vezes. Com a mesma semente, as repetições têm de dar **o mesmo modelo** (conferido). O tempo é reportado como mediana, mínimo e máximo, e o pico de memória como o maior. O modelo guardado é o da 1ª repetição.

**Aviso:** o modelo final foi treinado com os BENIGN da validação, então **não se mede qualidade nele com a validação**. A qualidade do modelo final aparece só no teste (notebook 06b).

In [11]:
# ---------- 6.1 Refit ----------
reps, modelo_final, p_ref = [], None, None
for k in range(1, REPETICOES_TEMPO + 1):
    with Medidor() as m:
        mdl = RandomForestClassifier(**HIPERPARAMETROS).fit(X_rf, y_rf)
    reps.append({"repeticao": k, "relogio_s": m.relogio, "cpu_s": m.cpu, "pico_ram_mib": m.pico_mib, "acrescimo_ram_mib": m.delta_mib})
    if modelo_final is None:
        modelo_final = mdl; modelo_final.colunas_modelo_ = list(CM); p_ref = mdl.predict_proba(X_va[:5000])[:, 1]
    else:
        assert np.allclose(mdl.predict_proba(X_va[:5000])[:, 1], p_ref), "as repetições do refit deram modelos diferentes (a semente não está fixando o resultado)"
    del mdl
tab_reps = pd.DataFrame(reps); display(tab_reps.round(3))
fit_refit = {"repeticoes": REPETICOES_TEMPO, "relogio_mediana_s": float(tab_reps.relogio_s.median()), "relogio_min_s": float(tab_reps.relogio_s.min()), "relogio_max_s": float(tab_reps.relogio_s.max()),
             "cpu_mediana_s": float(tab_reps.cpu_s.median()), "pico_ram_max_mib": float(tab_reps.pico_ram_mib.max()), "acrescimo_ram_max_mib": float(tab_reps.acrescimo_ram_mib.max())}
tam_fin = tamanho_e_nos(modelo_final, PASTA / "modelo_final_refit.joblib", COMPRESSAO_MODELO)
inf_fin = medir_inferencia(modelo_final, prep, X_va, X_va_raw, n_lat=N_LATENCIA, semente=SEMENTE)
print(f"\n✔ Refit: {fit_refit['relogio_mediana_s']:.1f} s (mediana de {REPETICOES_TEMPO}; mín {fit_refit['relogio_min_s']:.1f}, máx {fit_refit['relogio_max_s']:.1f}), CPU {fit_refit['cpu_mediana_s']:.1f} s, pico de RAM {fit_refit['pico_ram_max_mib']:,.0f} MiB")
print(f"Modelo final: {tam_fin['arquivo_mib']:.1f} MiB em disco | {tam_fin['nos_total']:,} nós | profundidade média {tam_fin['profundidade_media']:.1f} (máx {tam_fin['profundidade_max']})")
print("(latência e vazão do modelo final: medidas na validação só para VELOCIDADE; não são métricas de qualidade)")
display(pd.DataFrame(inf_fin).set_index("n_jobs").T.round(3))

,repeticao,relogio_s,cpu_s,pico_ram_mib,acrescimo_ram_mib
0,1,7.021,49.219,608.180,77.461
1,2,6.398,46.094,615.520,56.539
2,3,6.555,47.234,615.371,34.590



✔ Refit: 6.6 s (mediana de 3; mín 6.4, máx 7.0), CPU 47.2 s, pico de RAM 616 MiB
Modelo final: 0.8 MiB em disco | 40,214 nós | profundidade média 19.2 (máx 26)
(latência e vazão do modelo final: medidas na validação só para VELOCIDADE; não são métricas de qualidade)


n_jobs,1,-1
n_latencia,1000.000,1000.000
latencia_modelo_p50_ms,8.840,51.445
latencia_modelo_p95_ms,18.082,90.258
latencia_modelo_p99_ms,24.322,114.321
latencia_fim_a_fim_p50_ms,8.919,51.567
latencia_fim_a_fim_p95_ms,18.150,90.440
latencia_fim_a_fim_p99_ms,24.406,114.620
vazao_modelo_fluxos_s,137919.239,329296.278
vazao_fim_a_fim_fluxos_s,120441.521,252427.168
n_lote,60154.000,60154.000


## 7. Gravação dos registros

Grava os artefatos (modelos, configuração, pré-processamento, métricas, dispersão, validação cruzada, custo e limiares) e **acrescenta uma linha** a `registro_experimentos.csv` (um registro por execução). Em seguida **relê** o modelo gravado e confere que ele reproduz as mesmas previsões.

In [12]:
# ---------- 7.1 Gravação ----------
hw = {"so": platform.platform(), "python": sys.version.split()[0], "cpu": platform.processor(), "nucleos_logicos": os.cpu_count()}
if psutil: hw.update({"nucleos_fisicos": psutil.cpu_count(logical=False), "ram_total_gib": psutil.virtual_memory().total / 2 ** 30})
sw = {"scikit-learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__, "joblib": joblib.__version__, "psutil": None if psutil is None else psutil.__version__}
config = {"run_id": RUN_ID, "data": datetime.now().isoformat(timespec="seconds"), "modelo": "RandomForestClassifier", "variante": "com_auc", "hiperparametros": HIPERPARAMETROS, "semente": SEMENTE,
          "limiar_padrao": LIMIAR, "versao_selecao_notebook04": versao_sel, "versao_views_notebook03": versao_views, "razao_ataque_benigno": RAZAO_ATAQUE_BENIGNO,
          "dados": {"ajuste": len(X_aj), "validacao": len(X_va), "refit": len(X_rf), "benign_treino": N_BENIGN, "ataques_treino": N_ATAQUE, "peso_benign_proporcao_real": PESO_BENIGN_PROP},
          "protocolo": {"k_particoes_validacao": K_PARTICOES_VAL, "k_cv_ajuste": K_CV, "repeticoes_tempo_refit": REPETICOES_TEMPO, "n_latencia": N_LATENCIA, "fpr_alvos": list(FPR_ALVOS)},
          "software": sw, "hardware": hw}
salvar_json(config, PASTA / "config_treino.json"); salvar_json(prep.para_dict(), PASTA / "preprocessamento.json")
met_val = {"limiar": LIMIAR, "sem_pesos": m_sem, "proporcao_real": m_real, "pontos_de_operacao": PONTOS,
           "dispersao_particoes": {"k": K_PARTICOES_VAL, "resumo": resumo_part.to_dict(orient="index")}, "cv_ajuste": {"k": K_CV, "resumo": resumo_cv.to_dict(orient="index")}}
salvar_json(met_val, PASTA / "metricas_validacao.json")
pd.concat([tab_val.assign(limiar=LIMIAR)], axis=0).reset_index(names="metrica").to_csv(PASTA / "metricas_validacao.csv", index=False)
tab_part.to_csv(PASTA / "dispersao_particoes_validacao.csv", index=False); tab_cv.to_csv(PASTA / "cv_ajuste.csv", index=False)
salvar_json({"limiar_padrao": LIMIAR, "pontos_de_operacao_validacao": PONTOS}, PASTA / "limiares_validacao.json")
custo = {"fit_ajuste": fit_ajuste, "fit_refit": fit_refit, "fit_refit_repeticoes": reps, "modelo_ajuste": tam_aj, "modelo_final": tam_fin, "inferencia_modelo_ajuste": inf_aj, "inferencia_modelo_final": inf_fin}
salvar_json(custo, PASTA / "custo_computacional.json")
linhas_custo = [{"item": "fit_ajuste", **fit_ajuste}, {"item": "fit_refit", **{k: v for k, v in fit_refit.items()}}] + [{"item": f"inferencia_final_njobs{r['n_jobs']}", **r} for r in inf_fin] + [{"item": "modelo_final", **tam_fin}]
pd.DataFrame(linhas_custo).to_csv(PASTA / "custo_computacional.csv", index=False)

i1 = next(r for r in inf_fin if r["n_jobs"] == 1); iN = inf_fin[-1]
registro = {"run_id": RUN_ID, "data": config["data"], "modelo": "RandomForest", "hiperparametros": json.dumps(_limpar(HIPERPARAMETROS)), "n_ajuste": len(X_aj), "n_validacao": len(X_va), "n_refit": len(X_rf),
            "versao_selecao": versao_sel, "versao_views": versao_views, "roc_auc_val": m_sem["roc_auc"], "recall_val": m_sem["recall"], "fpr_val": m_sem["taxa_falso_alarme"],
            "precisao_real_val": m_real["precisao"], "f1_real_val": m_real["f1"], "mcc_real_val": m_real["mcc"], "pr_auc_real_val": m_real["pr_auc"], "f1_benign_real_val": m_real["f1_benign"], "pr_auc_benign_real_val": m_real["pr_auc_benign"], "recall_fpr1pct_val": PONTOS[0]["recall"] if PONTOS else float("nan"),
            "particoes_roc_auc_media": resumo_part.loc["roc_auc", "mean"], "particoes_roc_auc_dp": resumo_part.loc["roc_auc", "std"],
            "cv_roc_auc_media": resumo_cv.loc["roc_auc", "mean"] if len(resumo_cv) else float("nan"), "cv_roc_auc_dp": resumo_cv.loc["roc_auc", "std"] if len(resumo_cv) else float("nan"),
            "fit_ajuste_relogio_s": fit_ajuste["relogio_s"], "fit_ajuste_cpu_s": fit_ajuste["cpu_s"], "fit_ajuste_pico_ram_mib": fit_ajuste["pico_ram_mib"],
            "fit_refit_relogio_mediana_s": fit_refit["relogio_mediana_s"], "modelo_final_mib": tam_fin["arquivo_mib"], "nos_total": tam_fin["nos_total"],
            "latencia_p50_ms_njobs1": i1["latencia_fim_a_fim_p50_ms"], "latencia_p99_ms_njobs1": i1["latencia_fim_a_fim_p99_ms"], "vazao_fluxos_s_njobs1": i1["vazao_fim_a_fim_fluxos_s"],
            "vazao_fluxos_s_njobs_orig": iN["vazao_fim_a_fim_fluxos_s"], "pasta": str(PASTA)}
caminho_reg = ARTEFATOS_BASE / "registro_experimentos.csv"
pd.DataFrame([registro]).to_csv(caminho_reg, mode="a", header=not caminho_reg.exists(), index=False)
print(f"✔ Artefatos gravados em {PASTA}\n✔ Registro acrescentado a {caminho_reg} ({sum(1 for _ in open(caminho_reg, encoding='utf-8')) - 1} execução(ões) registrada(s))")
for f in sorted(PASTA.iterdir()): print(f"   {f.name:38s} {f.stat().st_size / 2**10:>10,.1f} KiB")

✔ Artefatos gravados em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_binario_com_auc\rf_bin_20261024_002027
✔ Registro acrescentado a C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_binario_com_auc\registro_experimentos.csv (1 execução(ões) registrada(s))
   config_treino.json                            1.2 KiB
   custo_computacional.csv                       1.1 KiB
   custo_computacional.json                      3.2 KiB
   cv_ajuste.csv                                 0.7 KiB
   dispersao_particoes_validacao.csv             0.7 KiB
   limiares_validacao.json                       0.7 KiB
   metricas_validacao.csv                        0.8 KiB
   metricas_validacao.json                       4.1 KiB
   modelo_ajuste.joblib                        692.0 KiB
   modelo_final_refit.joblib                   797.2 KiB
   preprocessamento.json                         2.4 KiB
   refit_ids.csv                             3,868.5 KiB


## 8. Conferências finais
Relê os arquivos gravados e confere que o modelo salvo reproduz as previsões, que a ordem dos atributos e as medianas gravadas são as usadas, e que as contas dos conjuntos fecham.

In [13]:
# ---------- 8.1 Conferências ----------
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)

m_ajuste_lido = joblib.load(PASTA / "modelo_ajuste.joblib"); m_final_lido = joblib.load(PASTA / "modelo_final_refit.joblib")
verifica("Modelo do ajuste gravado reproduz as previsões da validação", np.allclose(m_ajuste_lido.predict_proba(X_va)[:, 1], p_va))
verifica("Modelo final gravado reproduz as previsões do modelo em memória", np.allclose(m_final_lido.predict_proba(X_va[:5000])[:, 1], p_ref))
prep_lido = json.loads((PASTA / "preprocessamento.json").read_text(encoding="utf-8"))
verifica("Ordem dos atributos gravada = ordem usada no treino e dentro do modelo", prep_lido["colunas_modelo"] == CM == list(m_final_lido.colunas_modelo_) == list(m_ajuste_lido.colunas_modelo_))
verifica("Medianas gravadas = medianas da tabela de auditoria do notebook 03b", prep_lido["medianas_treino"] == MEDIANAS and set(prep_lido["colunas_retiradas_ablacao"]) == set(RETIRADAS))
verifica("Hiperparâmetros do modelo = configuração escolhida", all(getattr(m_final_lido, k) == v for k, v in HIPERPARAMETROS.items()))
verifica("Ajuste balanceado 1:1", int((y_aj == 0).sum()) == int((y_aj == 1).sum()) == n_aj_b, f"({int((y_aj == 0).sum()):,} + {int((y_aj == 1).sum()):,})")
verifica("Refit com todos os BENIGN e balanceado", int((y_rf == 0).sum()) == N_BENIGN and int((y_rf == 1).sum()) == int(round(RAZAO_ATAQUE_BENIGNO * N_BENIGN)), f"({int((y_rf == 0).sum()):,} + {int((y_rf == 1).sum()):,})")
verifica("Refit contém todo o ajuste", set(aj["id_linha"]) <= set(rf_df["id_linha"]))
verifica("Refit: a view e a tabela de seleção têm as mesmas linhas", len(rf_df) == n_linhas(TAB_REFIT) == len(refit_sel))
if existe("prep_bal_bin_refit_sel"):
    _r = set(consultar("SELECT toString(id_linha) AS id FROM prep_bal_bin_refit_sel")["id"])
    verifica("Mesmo refit do notebook 05 (mesmas linhas): comparação justa com e sem os atributos de AUC alto", _r == set(rf_df["id_linha"]), f"({len(_r):,} linhas)")
if existe("prep_bal_bin_selecao"):
    _s = consultar("SELECT toString(id_linha) AS id, papel FROM prep_bal_bin_selecao")
    verifica("Mesmo ajuste e mesma validação do notebook 04 (mesmas linhas)", set(_s[_s.papel == "treino"]["id"]) == set(aj["id_linha"]) and set(_s[_s.papel == "validacao"]["id"]) == set(va["id_linha"]))
verifica("Probabilidades entre 0 e 1 e matriz de confusão fecha", bool(((p_va >= 0) & (p_va <= 1)).all()) and abs(m_sem["tp"] + m_sem["fp"] + m_sem["tn"] + m_sem["fn"] - len(y_va)) < 1e-9)
verifica("Validação ponderada tem a proporção BENIGN:ataque do treino", abs(float(w_prop[y_va == 0].sum()) / float(w_prop[y_va == 1].sum()) - N_BENIGN / N_ATAQUE) < 1e-9)
verifica("Métricas que não dependem da proporção são iguais nas duas colunas", all(abs(m_sem[k] - m_real[k]) < 1e-9 for k in ("roc_auc", "recall", "taxa_falso_alarme", "especificidade", "acuracia_balanceada")))
verifica("Arquivos esperados gravados", all((PASTA / f).exists() for f in ("modelo_ajuste.joblib", "modelo_final_refit.joblib", "config_treino.json", "preprocessamento.json", "metricas_validacao.json", "metricas_validacao.csv",
                                                                           "dispersao_particoes_validacao.csv", "cv_ajuste.csv", "custo_computacional.json", "custo_computacional.csv", "limiares_validacao.json", "refit_ids.csv")))
verifica("Custo medido: tempos positivos" + ("" if psutil else " (memória não medida: psutil ausente)"), fit_ajuste["relogio_s"] > 0 and fit_refit["relogio_mediana_s"] > 0 and inf_fin[0]["vazao_fim_a_fim_fluxos_s"] > 0)
assert all(ok), "há checagens reprovadas acima"
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas.")
print(f"\nPróximo passo: notebook 06b (avaliação no teste), apontando para {PASTA.name}")

✔ Modelo do ajuste gravado reproduz as previsões da validação 
✔ Modelo final gravado reproduz as previsões do modelo em memória 
✔ Ordem dos atributos gravada = ordem usada no treino e dentro do modelo 
✔ Medianas gravadas = medianas da tabela de auditoria do notebook 03b 
✔ Hiperparâmetros do modelo = configuração escolhida 
✔ Ajuste balanceado 1:1 (40,614 + 40,614)
✔ Refit com todos os BENIGN e balanceado (50,768 + 50,768)
✔ Refit contém todo o ajuste 
  conexão instável (RemoteDisconnected: Remote end closed connection without response); nova tentativa em 5s...
✔ Refit: a view e a tabela de seleção têm as mesmas linhas 
✔ Mesmo refit do notebook 05 (mesmas linhas): comparação justa com e sem os atributos de AUC alto (101,536 linhas)
✔ Mesmo ajuste e mesma validação do notebook 04 (mesmas linhas) 
✔ Probabilidades entre 0 e 1 e matriz de confusão fecha 
✔ Validação ponderada tem a proporção BENIGN:ataque do treino 
✔ Métricas que não dependem da proporção são iguais nas duas colunas